# Assignment 4 — Evals Walkthrough

Step-by-step build of the eval harness for the RAG system from Assignment 3.

**Prerequisites:**
- The Chroma index is built (run `notebooks/03-rag-walkthrough.ipynb` first, or run `python -m assignment_03_rag.solution.ingest`).
- `OPENAI_API_KEY` set in `.env`.

Estimated cost of running the full notebook: ~$0.10 USD.

In [ ]:
import sys, json, importlib
from pathlib import Path

REPO_ROOT = Path.cwd().resolve()
while REPO_ROOT.name != 'openai-rag-evals-bootcamp' and REPO_ROOT.parent != REPO_ROOT:
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))

from dotenv import load_dotenv
load_dotenv(REPO_ROOT / '.env')
print('repo root:', REPO_ROOT)

## Step 1 — Look at the gold dataset

20 items spanning easy, multi-hop, adversarial, and out-of-scope. Eyeball the categories to develop intuition for what each metric should reveal.

In [ ]:
from collections import Counter

gold = [json.loads(l) for l in (REPO_ROOT / 'data' / 'eval' / 'gold_qa.jsonl').read_text().splitlines() if l.strip()]
print(f'{len(gold)} gold items')
print('by category:', Counter(it['category'] for it in gold))
print()
for it in gold[:3]:
    print(f"[{it['id']}] {it['question']}")
    print(f'  expected_sources: {it["expected_sources"]}')
    print()

## Step 2 — Run the RAG pipeline on the gold set

We populate each item with `retrieved_sources`, `retrieved_chunks`, and `answer_text`. This step is identical to what you'd do for any eval harness — separate 'run' from 'measure'.

In [ ]:
run_eval = importlib.import_module('assignment_04_evals.solution.run_eval')

evaluated = []
for it in gold:
    evaluated.append(run_eval.evaluate_item(it, top_k=5, threshold=0.30))
    print(f"  {it['id']:20s} retrieved {len(evaluated[-1]['retrieved_sources'])} chunks; answer starts: {evaluated[-1]['answer_text'][:60]!r}")

## Step 3 — Retrieval metrics

These are **deterministic and free**. They tell you whether the right chunks made it into the context window. If recall is low, no amount of generation prompt-tuning will save you.

In [ ]:
from assignment_04_evals.solution.metrics_retrieval import recall_at_k, mean_reciprocal_rank

print(f'Recall@1 = {recall_at_k(evaluated, 1):.2%}')
print(f'Recall@3 = {recall_at_k(evaluated, 3):.2%}')
print(f'Recall@5 = {recall_at_k(evaluated, 5):.2%}')
print(f'MRR@5    = {mean_reciprocal_rank(evaluated, 5):.3f}')

## Step 4 — LLM-as-judge metrics

Now we score each item for *faithfulness* and *answer relevancy*. Watch closely — note that the judge produces **reasoning before score**, enforced via Pydantic Structured Outputs.

In [ ]:
from assignment_04_evals.solution.metrics_judge import (
    FAITHFULNESS_RUBRIC, ANSWER_RELEVANCY_RUBRIC,
    faithfulness_score, answer_relevancy_score, JudgeOutput,
)
print(FAITHFULNESS_RUBRIC)
print('---')
print('Schema enforced on judge output:')
print(JudgeOutput.model_json_schema())

In [ ]:
sample = evaluated[0]
print(f"Question: {sample['question']}")
print(f"Answer:   {sample['answer_text'][:200]}...")
print()
f = faithfulness_score(sample)
r = answer_relevancy_score(sample)
print(f'Faithfulness: {f.score} — {f.reasoning[:200]}')
print(f'Relevancy:    {r.score} — {r.reasoning[:200]}')

In [ ]:
for it in evaluated:
    if 'faithfulness_score' in it:
        continue
    f = faithfulness_score(it)
    r = answer_relevancy_score(it)
    it['faithfulness_score'] = f.score
    it['faithfulness_reasoning'] = f.reasoning
    it['answer_relevancy_score'] = r.score
    it['answer_relevancy_reasoning'] = r.reasoning
    print(f"{it['id']:20s} F={f.score} R={r.score}")

## Step 5 — Per-category breakdown

**Per-category metrics matter more than the overall.** A green overall metric can hide a red category — exactly what we're looking for.

In [ ]:
from collections import defaultdict
import statistics

by_cat = defaultdict(list)
for it in evaluated:
    by_cat[it['category']].append(it)

print(f"{'category':14s} {'n':>3s} {'recall@5':>10s} {'F':>5s} {'R':>5s}")
for cat, its in by_cat.items():
    rec = recall_at_k(its, 5)
    f = statistics.mean(i['faithfulness_score'] for i in its)
    r = statistics.mean(i['answer_relevancy_score'] for i in its)
    print(f'{cat:14s} {len(its):3d} {rec:10.2%} {f:5.2f} {r:5.2f}')

## Step 6 — Inspect failures

The most valuable thing you can do with an eval suite. Find low-scoring items and ask *why*.

In [ ]:
for it in evaluated:
    if it['faithfulness_score'] < 4 or it['answer_relevancy_score'] < 4:
        print(f"=== {it['id']} ({it['category']}) F={it['faithfulness_score']} R={it['answer_relevancy_score']} ===")
        print(f"Q: {it['question']}")
        print(f"A: {it['answer_text']}")
        print(f"  Faithfulness reason: {it['faithfulness_reasoning'][:300]}")
        print(f"  Relevancy reason:    {it['answer_relevancy_reasoning'][:300]}")
        print()

## Step 7 — Persist the report

Persist every run as JSON so future iterations can be diffed.  
These reports are also the input the pytest regression suite reads.

In [ ]:
results = {
    'config_name': 'walkthrough',
    'top_k': 5,
    'threshold': 0.30,
    'timestamp': __import__('datetime').datetime.utcnow().isoformat() + 'Z',
    'items': evaluated,
    'metrics': {
        'overall': {
            'recall_at_1': recall_at_k(evaluated, 1),
            'recall_at_3': recall_at_k(evaluated, 3),
            'recall_at_5': recall_at_k(evaluated, 5),
            'mrr_at_5': mean_reciprocal_rank(evaluated, 5),
            'mean_faithfulness': statistics.mean(i['faithfulness_score'] for i in evaluated),
            'mean_answer_relevancy': statistics.mean(i['answer_relevancy_score'] for i in evaluated),
        },
        'by_category': {
            cat: {
                'n': len(its),
                'recall_at_5': recall_at_k(its, 5),
                'mean_faithfulness': statistics.mean(i['faithfulness_score'] for i in its),
                'mean_answer_relevancy': statistics.mean(i['answer_relevancy_score'] for i in its),
            } for cat, its in by_cat.items()
        },
    },
}
path = run_eval.write_report(results)
print(f'Saved to {path}')

## Step 8 — Run the regression tests

The same metrics, expressed as `pytest` assertions. This is what you wire into CI.

In [ ]:
import subprocess
result = subprocess.run(
    ['pytest', '-v', 'assignment_04_evals/solution/tests/'],
    cwd=str(REPO_ROOT),
    capture_output=True, text=True,
)
print(result.stdout[-2000:])
if result.returncode != 0:
    print('STDERR:', result.stderr[-1000:])

## What to do next

1. **Compare two configs.** Re-ingest with `--chunk-size 1000`, run the eval again, write `reports/comparison.md`.
2. **Inspect the OOS items.** Did the system refuse on all 3? If not, raise the threshold.
3. **Add a citation-correctness judge.** This is the highest-leverage stretch goal.
4. **Run the suite 3× and compute std-dev.** You'll see how noisy LLM judges really are.